imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from torchvision import transforms
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision import models


show information about dataset

In [ ]:
BCS_COLUMN_NAME = 'bcs_mean'

df = pd.read_csv("BCS_dataset/labels.csv")
print(f"number of total images: {len(df)}")

if len(df) > 0:
    plt.figure(figsize=(10, 6))
    sns.set_style("whitegrid")

    ax = sns.histplot(data=df, x=BCS_COLUMN_NAME, bins=20, kde=True, color='teal')

    plt.title(f'Distribution of Body Condition Score (BCS)\nTotal Samples: {len(df)}', fontsize=15)
    plt.xlabel('BCS (Exact Mean Values)', fontsize=12)
    plt.ylabel('Number of Images', fontsize=12)

    plt.xlim(2, 5)

    plt.show()

print(df[BCS_COLUMN_NAME].describe())
print(df[BCS_COLUMN_NAME].value_counts().sort_index())



removing corrupted files

In [ ]:
image_dir = "BCS_dataset/images"

corrupted_files = []
black_images = []
white_images = []

widths = []
heights = []

black_threshold = 5
white_threshold = 250

for filename in os.listdir(image_dir):
    path = os.path.join(image_dir, filename)

    if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        continue

    try:
        img = Image.open(path)

        width, height = img.size
        widths.append(width)
        heights.append(height)

        img_array = np.array(img)

        mean_pixel = img_array.mean()

        if mean_pixel < black_threshold:
            black_images.append(filename)
        elif mean_pixel > white_threshold:
            white_images.append(filename)

    except:
        corrupted_files.append(filename)

print(f"number of corrupted images: {len(corrupted_files)}")
print(f"number of black images: {len(black_images)}")
print(f"number of white images: {len(white_images)}")

print(f"range of width of images: {list(set(widths))}")
print(f"range of height of images: {list(set(heights))}")

# width distribution

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.hist(widths, bins=20)
plt.title("Distribution of Image Width")
plt.xlabel("Width (pixels)")
plt.ylabel("Number of Images")
plt.grid(alpha=0.3)

# height distribution 
plt.subplot(1, 2, 2)
plt.hist(heights, bins=20)
plt.title("Distribution of Image Height")
plt.xlabel("Height (pixels)")
plt.ylabel("Number of Images")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



splitting

In [ ]:
df['bcs_bin'] = pd.qcut(df[BCS_COLUMN_NAME], 4, labels=False, duplicates='drop')
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['bcs_bin'])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['bcs_bin'])

train_df.drop(columns=['bcs_bin'], inplace=True)
test_df.drop(columns=['bcs_bin'], inplace=True)
val_df.drop(columns=['bcs_bin'], inplace=True)

train_df.to_csv('BCS_dataset/train.csv', index=False)
val_df.to_csv("BCS_dataset/val.csv", index=False)
test_df.to_csv("BCS_dataset/test.csv", index=False)

print(len(train_df))
print(len(val_df))
print(len(test_df))

# distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# Train
sns.histplot(train_df[BCS_COLUMN_NAME], bins=20, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Train Distribution')
axes[0].set_xlabel(BCS_COLUMN_NAME)
axes[0].set_ylabel('Count')

# Validation
sns.histplot(val_df[BCS_COLUMN_NAME], bins=20, kde=True, ax=axes[1], color='green')
axes[1].set_title('Validation Distribution')
axes[1].set_xlabel(BCS_COLUMN_NAME)
axes[1].set_ylabel('Count')

# Test
sns.histplot(test_df[BCS_COLUMN_NAME], bins=20, kde=True, ax=axes[2], color='red')
axes[2].set_title('Test Distribution')
axes[2].set_xlabel(BCS_COLUMN_NAME)
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()


transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),

    transforms.ToTensor(),
    # mean and std of ImageNet used in ResNet
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Datasets

In [ ]:
class BCSDataset(Dataset):
    def __init__(self, df, transform):
        self.data_frame = df
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_path = self.data_frame.iloc[idx]['local_path']
        label = self.data_frame.iloc[idx][BCS_COLUMN_NAME]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


train_dataset = BCSDataset(train_df, train_transform)
test_dataset = BCSDataset(test_df, test_val_transform)
val_dataset = BCSDataset(val_df, test_val_transform)


Dataloaders

In [ ]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

loading ResNet model and changing last layers in model

In [ ]:
class BCSResNet18(nn.Module):

    def __init__(self, trainable_layers="fc"):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT
        self.model = models.resnet18(weights=weights)
        self.model.fc = nn.Linear(in_features=512, out_features=1)

        # Freeze all pretrained layers
        for param in self.model.parameters():
            param.requires_grad = False
        # unfreeze trainable layers
        self._set_trainable_layers(trainable_layers)

    def _set_trainable_layers(self, trainable_layers):
        if trainable_layers == "fc":

            for param in self.model.fc.parameters():
                param.requires_grad = True

        elif trainable_layers == "layer4_fc":

            for param in self.model.layer4.parameters():
                param.requires_grad = True

            for param in self.model.fc.parameters():
                param.requires_grad = True

        elif trainable_layers == "layer3_layer4_fc":

            for param in self.model.layer3.parameters():
                param.requires_grad = True

            for param in self.model.layer4.parameters():
                param.requires_grad = True

            for param in self.model.fc.parameters():
                param.requires_grad = True

        else:
            raise ValueError(
                f"Unknown trainable_layers: {trainable_layers}"
            )

    def forward(self, x):
        return self.model(x)






trainer

In [ ]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, device, optimizer, learning_rate, criterion=nn.MSELoss,
                 epoch=30, patience=5):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.learning_rate = learning_rate
        self.criterion = criterion
        self.epoch = epoch
        self.patience = patience
        self.optimizer = optimizer
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_mae': [], 'val_mae': []
        }

    def train_one_epoch(self):
        self.model.train()  # putting on training mode

        total_loss = 0.0
        total_absolute_error = 0.0
        total_samples = 0

        for images, labels in self.train_loader:  # giving all images and labels in a batch
            images, labels = images.to(self.device), labels.to(self.device)

            # forward pass
            predictions = self.model(images)
            predictions = predictions.squeeze(1)  # converting to tensor with 1 dimension

            # loss
            loss = self.criterion(predictions, labels)

            # backward
            self.optimizer.zero_grad()
            loss.backward()

            self.optimizer.step()

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_absolute_error += torch.abs(predictions - labels).sum().item()
            total_samples += batch_size

        epoch_loss = total_loss / total_samples
        epoch_mae = total_absolute_error / total_samples
        
        return epoch_loss, epoch_mae
    
    def validate(self):
        self.model.eval()
        
        total_loss = 0.0
        total_absolute_error = 0.0
        total_samples = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader: 
                images, labels = images.to(self.device), labels.to(self.device)
                
                predictions = self.model(images)
                predictions = predictions.squeeze(1)
                val_loss = self.criterion(predictions, labels)
                
                batch_size = images.size(0)
    
                total_loss += val_loss.item() * batch_size
                total_absolute_error += torch.abs(predictions - labels).sum().item()
                total_samples += batch_size

        epoch_loss = total_loss / total_samples
        epoch_mae = total_absolute_error / total_samples
        
        return epoch_loss, epoch_mae
        